# Text Summarization - 1A
## Artikel: BPI Danantara & Qatar Investment
**Metode:** TF-IDF dengan Sastrawi (Bahasa Indonesia)

## 1. Install & Import Dependencies

In [ ]:
!pip install Sastrawi -q

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import matplotlib.pyplot as plt
import pandas as pd

nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)
print('Dependencies loaded successfully')

## 2. Define Document

In [ ]:
article = """
Jakarta: Badan Pengelola Investasi Daya Anagata Nusantara (BPI Danantara) siap mengawal realisasi investasi yang telah disepakati dengan Qatar. Kesepakatan antara Indonesia dan Qatar merupakan buah dari kunjungan resmi Presiden Prabowo Subianto ke Doha.

Pemerintah Republik Indonesia dan Pemerintah Qatar menggelar diskusi untuk menyepakati kemitraan strategis (co-partnership) dalam pengelolaan dana investasi untuk Indonesia yang akan berfokus di berbagai sektor pembangunan.

Salah satu hasil utama dari kunjungan tersebut adalah untuk membentuk dana investasi bersama senilai USD4 miliar. Dana ini akan difokuskan pada pengembangan berbagai sektor di antaranya termasuk tapi tidak terbatas pada hilirisasi industri, energi terbarukan, dan fasilitas kesehatan di Indonesia.

Kami menyambut baik kepercayaan yang diberikan oleh Pemerintah Qatar melalui pembentukan dana bersama ini, kata CEO Danantara Indonesia Rosan Perkasa Roeslani dalam keterangan tertulis, Selasa, 15 April 2025.

Presiden Prabowo menyampaikan masing-masing negara akan berkontribusi sebesar USD2 miliar dalam dana tersebut. Dana itu akan dikelola oleh BPI Danantara bersama dengan Qatar Investment Authority (QIA) dalam co-partnership.

Dana tersebut akan difokuskan pada peluang investasi di berbagai sektor strategis, antara lain hilirisasi, kesehatan, energi terbarukan, teknologi, serta sektor-sektor lain yang dipandang relevan oleh pengelola dana.

Danantara Indonesia siap menjalankan mandat tersebut dengan menerapkan tata kelola investasi yang prudent, transparan, dan berorientasi pada hasil. Fokus kami adalah memastikan bahwa setiap proyek yang didanai memberikan dampak strategis dan berkelanjutan bagi perekonomian nasional, ujar Rosan.

Lebih lanjut, Rosan menegaskan, kolaborasi ini menjadi bukti kepercayaan dunia internasional terhadap kapasitas kelembagaan Indonesia dalam mengelola investasi berskala besar.

Kemitraan ini merupakan langkah konkret dalam membangun kepercayaan dengan mitra global strategis seperti Qatar. Ini menunjukkan bahwa Indonesia tidak hanya menjadi tujuan investasi, tetapi juga memiliki kapasitas kelembagaan yang mumpuni untuk mengelola investasi secara profesional dan akuntabel.

Inisiatif co-partnership dan perluasan kerja sama strategis ini diharapkan tidak hanya memperkuat hubungan diplomatik kedua negara, tetapi juga memberikan kontribusi nyata terhadap percepatan pembangunan ekonomi dan peningkatan kesejahteraan masyarakat Indonesia.
"""

print(f'Article length: {len(article)} characters')

## 3. Text Preprocessing

In [ ]:
# Sentence tokenization
sent_tokens = sent_tokenize(article)

print(f'Total sentences: {len(sent_tokens)}\n')
print('=== List of Sentences ===')
for i, s in enumerate(sent_tokens):
    print(f'{i+1}. {s}')

In [ ]:
# Remove Indonesian stop words using Sastrawi
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

cleaned_sentences = [stopword_remover.remove(s) for s in sent_tokens]

print('=== Cleaned Sentences (stop words removed) ===')
for i, s in enumerate(cleaned_sentences):
    print(f'{i+1}. {s}')

## 4. TF-IDF Vectorization

In [ ]:
# Build TF-IDF matrix
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(cleaned_sentences)
feature_names = vectorizer.get_feature_names_out()

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Total sentences: {tfidf_matrix.shape[0]}, Total unique words: {tfidf_matrix.shape[1]}')

In [ ]:
# Show top TF-IDF words for the first sentence
df_first = pd.DataFrame({
    'Word': feature_names,
    'TF-IDF': tfidf_matrix[0].toarray().flatten()
})
df_first = df_first[df_first['TF-IDF'] > 0].sort_values('TF-IDF', ascending=False)

print('=== Top TF-IDF words in Sentence 1 ===')
print(df_first.to_string(index=False))

## 5. Sentence Scoring

In [ ]:
# Calculate average TF-IDF score per sentence
sent_scores = []
for i, row in enumerate(tfidf_matrix):
    total = row.sum()
    n_words = len(row.data)
    avg = total / n_words if n_words > 0 else 0
    sent_scores.append(avg)

print('=== Average TF-IDF Score per Sentence ===')
for i, score in enumerate(sent_scores):
    print(f'Sentence {i+1}: {score:.4f}')

In [ ]:
# Visualize sentence scores
plt.figure(figsize=(12, 5))
plt.bar(range(1, len(sent_scores)+1), sent_scores, color='steelblue')
plt.xlabel('Sentence Number')
plt.ylabel('Average TF-IDF Score')
plt.title('Average TF-IDF Score per Sentence\n(Artikel: BPI Danantara & Qatar)')
plt.xticks(range(1, len(sent_scores)+1))
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 6. Text Summarization

In [ ]:
# Threshold = mean score
threshold = sum(sent_scores) / len(sent_scores)
print(f'Threshold (mean score): {threshold:.4f}\n')

summary_sentences = []
print('=== Sentences selected for summary ===')
for i, (sent, score) in enumerate(zip(sent_tokens, sent_scores)):
    if score >= threshold:
        summary_sentences.append(sent)
        print(f'[v] Sentence {i+1} (score={score:.4f}): {sent[:80]}...')

In [ ]:
# Final summary
final_summary = ' '.join(summary_sentences)

print('=== RINGKASAN ARTIKEL ===')
print(final_summary)